In [0]:
dbutils.widgets.text("catalog","sky_dev")
catalog = dbutils.widgets.get("catalog")

dbutils.widgets.text("job_id","ADHOC")
job_id = dbutils.widgets.get("job_id")

In [0]:
print(job_id)

### Load Passenger

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.passengers
""")

In [0]:
spark.sql(f"""
copy into {catalog}.bronze.passengers
from (select *, _metadata.file_path as file_path, current_timestamp() as ingestion_ts, '{job_id}' as job_run_id from '/Volumes/{catalog}/bronze/raw/passenger')
fileformat = csv
format_options ('header' = 'true', 'inferSchema' = 'true')
copy_options ('mergeSchema' = 'true')
""")

In [0]:
%sql
select count(*) from sky_dev.bronze.passengers

### Load region

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.regions
""")

In [0]:
spark.sql(f"""
copy into {catalog}.bronze.regions
from (select *, _metadata.file_path as file_path, current_timestamp() as ingestion_ts, '{job_id}' as job_run_id from '/Volumes/{catalog}/bronze/raw/region')
fileformat = csv
format_options ('header' = 'true', 'inferSchema' = 'true')
copy_options ('mergeSchema' = 'true')
""")

In [0]:
%sql
select count(*) from sky_dev.bronze.regions

### Load Nations

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.nations
""")

In [0]:
spark.sql(f"""
copy into {catalog}.bronze.nations
from (select *, _metadata.file_path as file_path, current_timestamp() as ingestion_ts, '{job_id}' as job_run_id from '/Volumes/{catalog}/bronze/raw/nation')
fileformat = csv
format_options ('header' = 'true', 'inferSchema' = 'true')
copy_options ('mergeSchema' = 'true')
""")

In [0]:
%sql
select count(*) from sky_dev.bronze.nations

### Load airline routes

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.airline_routes
""")

In [0]:
spark.sql(f"""
copy into {catalog}.bronze.airline_routes
from (select *, _metadata.file_path as file_path, current_timestamp() as ingestion_ts, '{job_id}' as job_run_id from '/Volumes/{catalog}/bronze/raw/airline_route')
fileformat = csv
format_options ('header' = 'true', 'inferSchema' = 'true')
copy_options ('mergeSchema' = 'true')
""")

In [0]:
%sql
select count(*) from sky_dev.bronze.airline_routes

### Load Airlines

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.airlines
""")

In [0]:
spark.sql(f"""
copy into {catalog}.bronze.airlines
from (select *, _metadata.file_path as file_path, current_timestamp() as ingestion_ts, '{job_id}' as job_run_id from '/Volumes/{catalog}/bronze/raw/airline')
fileformat = csv
format_options ('header' = 'true', 'inferSchema' = 'true')
copy_options ('mergeSchema' = 'true')
""")

In [0]:
%sql
select count(*) from sky_dev.bronze.airlines

### Load Routes

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.routes
""")

In [0]:
spark.sql(f"""
copy into {catalog}.bronze.routes
from (select *, _metadata.file_path as file_path, current_timestamp() as ingestion_ts, '{job_id}' as job_run_id from '/Volumes/{catalog}/bronze/raw/route')
fileformat = csv
format_options ('header' = 'true', 'inferSchema' = 'true')
copy_options ('mergeSchema' = 'true')
""")

In [0]:
%sql
select count(*) from sky_dev.bronze.routes

### Load Bookings

In [0]:
from pyspark.sql import functions as F
source_path = f"/Volumes/{catalog}/bronze/raw/booking"
checkpoint_path = f"/Volumes/{catalog}/bronze/pipeline_state/checkpoint/bookings"
schema_path = f"/Volumes/{catalog}/bronze/pipeline_state/schema/bookings"
df = spark.readStream.format('cloudFiles') \
    .option('cloudFiles.format','csv') \
    .option('cloudFiles.inferColumnTypes','true') \
    .option('cloudFiles.schemaLocation', schema_path) \
    .option('cloudFiles.schemaEvolutionMode', 'addNewColumns') \
    .option('cloudFiles.rescuedDataColumn', '_rescued_data') \
    .option('header','true') \
    .load(source_path)

df = df.withColumns({'ingestion_ts': F.current_timestamp(),
                     'file_path' : F.col("_metadata.file_path"),
                     'job_run_id': F.lit(job_id)
    })

df.writeStream.format('delta') \
    .option('checkpointLocation', checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(f'{catalog}.bronze.bookings')

In [0]:
%sql
select count(*) from sky_dev.bronze.bookings

### Load flight bookings

In [0]:
from pyspark.sql import functions as F
source_path = f"/Volumes/{catalog}/bronze/raw/flight_booking"
checkpoint_path = f"/Volumes/{catalog}/bronze/pipeline_state/checkpoint/flight_bookings"
schema_path = f"/Volumes/{catalog}/bronze/pipeline_state/schema/flight_bookings"
df = spark.readStream.format('cloudFiles') \
    .option('cloudFiles.format','csv') \
    .option('cloudFiles.inferColumnTypes','true') \
    .option('cloudFiles.schemaLocation', schema_path) \
    .option('cloudFiles.schemaEvolutionMode', 'addNewColumns') \
    .option('cloudFiles.rescuedDataColumn', '_rescued_data') \
    .option('header','true') \
    .load(source_path)

df = df.withColumns({'ingestion_ts': F.current_timestamp(),
                     'file_path' : F.col("_metadata.file_path"),
                     'job_run_id': F.lit(job_id)
    })

df.writeStream.format('delta') \
    .option('checkpointLocation', checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(f'{catalog}.bronze.flight_bookings')

In [0]:
%sql
select count(*) from sky_dev.bronze.flight_bookings